# Run FilterZyme on sequences form Yang et al. 

## Prep input data

In [ ]:
import pandas as pd
import numpy as np
import re

def clean_sequence(seq: str) -> str:

    if pd.isna(seq):
        return None
    seq = seq.upper()
    seq = re.sub(r'[^ACDEFGHIKLMNPQRSTVWY]', '', seq)  # Keep only standard 20 amino acids
    return seq

# Load sequence and activity data from Yang et al. 
sequence_activity_data = pd.read_excel("Yangetal_Experimental_results_tabulation.xlsx", sheet_name=None)

# Process CuSOD data
round1_CuSOD = sequence_activity_data['round1_CuSOD']
round1_CuSOD['round'] = 'round1'
round2_CuSOD = sequence_activity_data['round2_CuSOD']
round2_CuSOD = round2_CuSOD.iloc[:, :-5]
round2_CuSOD['round'] = 'round2'
round3_CuSOD = sequence_activity_data['round3_CuSOD']
round3_CuSOD = round3_CuSOD.iloc[:, :-5]
round3_CuSOD['round'] = 'round3'

for df_name, df in (("r1", round1_CuSOD), ("r2", round2_CuSOD), ("r3", round3_CuSOD)):
    df.columns = df.columns.astype(str).str.strip()
    df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)
    # keep first occurrence of any duplicated column label
    df = df.loc[:, ~df.columns.duplicated()]
    if df_name == "r1":
        round1_MDH = df
    elif df_name == "r2":
        round2_MDH = df
    else:
        round3_MDH = df

CuSOD_all_rounds = pd.concat([round1_CuSOD, round2_CuSOD, round3_CuSOD], ignore_index=True)

# Process MDH data
round1_MDH = sequence_activity_data['round1_MDH']
round1_MDH['round'] = 'round1'
round2_MDH = sequence_activity_data['round2_MDH']
round2_MDH = round2_MDH.iloc[:, :-5]
round2_MDH['round'] = 'round2'  
round3_MDH = sequence_activity_data['round3_MDH']
round3_MDH = round3_MDH.iloc[:, :-5]
round3_MDH['round'] = 'round3'

for df_name, df in (("r1", round1_MDH), ("r2", round2_MDH), ("r3", round3_MDH)):
    df.columns = df.columns.astype(str).str.strip()
    df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)
    # keep first occurrence of any duplicated column label
    df = df.loc[:, ~df.columns.duplicated()]
    if df_name == "r1":
        round1_MDH = df
    elif df_name == "r2":
        round2_MDH = df
    else:
        round3_MDH = df

MDH_all_rounds = pd.concat([round1_MDH, round2_MDH, round3_MDH], ignore_index=True)

# Only consider purified proteins for activity analysis
CuSOD_all_rounds = CuSOD_all_rounds[CuSOD_all_rounds['Purification'] == 'Y'].copy()
CuSOD_all_rounds = CuSOD_all_rounds.rename(columns={'Activity (5μg/mL)': 'Activity', "expressed sequence": "Sequence"})

MDH_all_rounds = MDH_all_rounds[MDH_all_rounds['Purification'] == 'Y'].copy()
MDH_all_rounds = MDH_all_rounds.rename(columns={'Activity (20 μg/mL)': 'Activity', "expressed sequence": "Sequence"})

# Add all necessary columns
CuSOD_all_rounds["enzyme"] = "CuSOD"
CuSOD_all_rounds['substrate_name'] = "superoxide_radical"
CuSOD_all_rounds['substrate_smiles'] = "[O-][O]" # Superoxide radical
CuSOD_all_rounds['cofactor_name'] = [[] for _ in range(len(CuSOD_all_rounds))]
CuSOD_all_rounds['cofactor_smiles'] = [[] for _ in range(len(CuSOD_all_rounds))]
CuSOD_all_rounds['substrate_moiety'] = None
CuSOD_all_rounds['cofactor_moiety'] = None

MDH_all_rounds["enzyme"] = "MDH"
MDH_all_rounds['substrate_name'] = "oxaloacetic_acid"
MDH_all_rounds['substrate_smiles'] ="O=C(O)C(=O)CC(=O)O" # Oxaloacetic acid
MDH_all_rounds['cofactor_name'] = "NADH"
MDH_all_rounds['cofactor_smiles'] = "C1C=CN(C=C1C(=O)N)[C@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)N4C=NC5=C(N=CN=C54)N)O)O)O)O" # NADH
MDH_all_rounds['substrate_moiety'] = "O=C(C(=O)O)CC(=O)"  # Malonate moiety
MDH_all_rounds['cofactor_moiety'] = "NC(=O)[n+]1ccc(C)[CH-]1"  # Nicotinamide moiety

# Combine the filtered datasets
MDH_CuSOD_combined = pd.concat([CuSOD_all_rounds, MDH_all_rounds], ignore_index=True)
MDH_CuSOD_combined['Sequence'] = MDH_CuSOD_combined['Sequence'].apply(clean_sequence)
MDH_CuSOD_combined['Entry'] = (
    MDH_CuSOD_combined['Name'].astype(str) + "_" + MDH_CuSOD_combined['enzyme'].astype(str)
)
MDH_CuSOD_combined['Entry'] = MDH_CuSOD_combined['Entry'].str.replace('.', 'p', regex=False)

# Round 3 data only
MDH_CuSOD_round3 = MDH_CuSOD_combined[MDH_CuSOD_combined['round'] == 'round3'].copy()

# MDH_CuSOD_combined.to_pickle('MDH_CuSOD_combined_input_df.pkl')
# MDH_CuSOD_round3.to_pickle('MDH_CuSOD_round3_input_df.pkl')

/home/helen/miniconda3/envs/promiscuity/lib/python3.12/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
